<a href="https://colab.research.google.com/github/ysuter/FHNW-BAI-ComputerVision/blob/main/foundation_models_woche14_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Foundation Models in Computer Vision – Woche 14
##Demo: DINOv3 als Feature-Extraktor

**Inhalt**
1. Setup und Imports
2. DINOv3 laden (eingefroren)
3. Eigene Bilder einbetten
4. Embedding-Raum visualisieren (UMAP/PCA)
5. k-NN-Bildsuche
6. Vergleich: DINOv3 vs. ImageNet-vortrainiertes ResNet50
7. Bonus: Attention Maps anzeigen

> **Wichtig:** DINOv3 ist auf Hugging Face *gated* – ihr müsst auf der Modellseite einmalig den Zugang anfragen (kommt sofort) und in Colab via `huggingface_hub.login()` einloggen.

> **Plattform:** läuft sauber in Google Colab (CPU oder GPU). Mit ViT-S/16 reicht eine CPU-Sitzung.

## 1. Setup und Imports

DINOv3 benötigt `transformers >= 4.56.0`. Wir installieren ausserdem UMAP für die Visualisierung und scikit-learn für PCA und k-NN.

In [ ]:
# In Colab: Installation
!pip install --upgrade torch torchvision transformers umap-learn scikit-learn matplotlib pillow

In [ ]:
import os
import io
import math
import urllib.request
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image
import matplotlib.pyplot as plt
from transformers import AutoImageProcessor, AutoModel

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
print(f"PyTorch: {torch.__version__}")

### Hugging Face Login

DINOv3-Modelle sind gated. Einmalig auf der Modellseite Zugang beantragen
(z.B. `facebook/dinov3-vits16-pretrain-lvd1689m`), dann hier einloggen:

In [ ]:
from huggingface_hub import login
login()  # Token aus https://huggingface.co/settings/tokens

## 2. DINOv3 laden (eingefroren)

Wir nehmen die kleine ViT-S/16-Variante (~21 M Parameter) – schnell genug für Colab und gute Qualität für unsere Demos.

Verfügbare Varianten:
- `facebook/dinov3-vits16-pretrain-lvd1689m` (klein, ~21 M)
- `facebook/dinov3-vitb16-pretrain-lvd1689m` (mittel, ~86 M)
- `facebook/dinov3-vitl16-pretrain-lvd1689m` (gross, ~300 M)
- `facebook/dinov3-convnext-tiny-pretrain-lvd1689m` (CPU-freundlich)

In [ ]:
import transformers, os
print("transformers:", transformers.__version__)

from huggingface_hub import HfApi
try:
    info = HfApi().model_info("facebook/dinov3-vits16-pretrain-lvd1689m")
    print("Auth OK, model accessible. Files:", [s.rfilename for s in info.siblings][:5])
except Exception as e:
    print("HF access problem:", e)

print("Local shadow dir exists?", os.path.exists("facebook/dinov3-vits16-pretrain-lvd1689m"))

In [ ]:
from transformers import AutoImageProcessor, AutoModel

MODEL_ID = "facebook/dinov3-vits16-pretrain-lvd1689m"

processor = AutoImageProcessor.from_pretrained(MODEL_ID)
model = AutoModel.from_pretrained(MODEL_ID).to(device)
model.eval()  # eingefroren – keine Gradienten

# Wie gross sind die Embeddings?
n_params = sum(p.numel() for p in model.parameters())
print(f"Modell: {MODEL_ID}")
print(f"Parameter: {n_params/1e6:.1f} M")
print(f"Hidden size: {model.config.hidden_size}")  # = Embedding-Dimension

## 3. Bilder laden und einbetten

Für die Demo nutzen wir ein paar Beispielbilder. Für eigene Experimente: Upload einrichten (per `files.upload()` in Colab oder Drag & Drop in Dateien).

In [ ]:
# Beispielbilder von freien Quellen (5 pro Kategorie)
SAMPLE_URLS = {
    # Katzen
    "katze_1":  "https://images.unsplash.com/photo-1574144611937-0df059b5ef3e?w=400",
    "katze_2":  "https://images.unsplash.com/photo-1543852786-1cf6624b9987?w=400",
    "katze_3":  "https://images.unsplash.com/photo-1514888286974-6c03e2ca1dba?w=400",
    "katze_4":  "https://images.unsplash.com/photo-1518791841217-8f162f1e1131?w=400",
    "katze_5":  "https://images.unsplash.com/photo-1592194996308-7b43878e84a6?w=400",

    # Hunde
    "hund_1":   "https://images.unsplash.com/photo-1587300003388-59208cc962cb?w=400",
    "hund_2":   "https://images.unsplash.com/photo-1561037404-61cd46aa615b?w=400",
    "hund_3":   "https://images.unsplash.com/photo-1517849845537-4d257902454a?w=400",
    "hund_4":   "https://images.unsplash.com/photo-1543466835-00a7907e9de1?w=400",
    "hund_5":   "https://images.unsplash.com/photo-1583337130417-3346a1be7dee?w=400",

    # Autos
    "auto_1":   "https://images.unsplash.com/photo-1494976388531-d1058494cdd8?w=400",
    "auto_2":   "https://images.unsplash.com/photo-1517676109075-9a94d44145d1?w=400",
    "auto_3":   "https://images.unsplash.com/photo-1503376780353-7e6692767b70?w=400",
    "auto_4":   "https://images.unsplash.com/photo-1502877338535-766e1452684a?w=400",
    "auto_5":   "https://images.unsplash.com/photo-1492144534655-ae79c964c9d7?w=400",

    # Pizza
    "pizza_1":  "https://images.unsplash.com/photo-1565299624946-b28f40a0ae38?w=400",
    "pizza_2":  "https://images.unsplash.com/photo-1513104890138-7c749659a591?w=400",
    "pizza_3":  "https://images.unsplash.com/photo-1574071318508-1cdbab80d002?w=400",
    "pizza_4":  "https://images.unsplash.com/photo-1604068549290-dea0e4a305ca?w=400",
    "pizza_5":  "https://images.unsplash.com/photo-1571066811602-716837d681de?w=400",

    # Berge
    "berge_1":  "https://images.unsplash.com/photo-1464822759023-fed622ff2c3b?w=400",
    "berge_2":  "https://images.unsplash.com/photo-1486870591958-9b9d0d1dda99?w=400",
    "berge_3":  "https://images.unsplash.com/photo-1454496522488-7a8e488e8606?w=400",
    "berge_4":  "https://images.unsplash.com/photo-1519681393784-d120267933ba?w=400",
    "berge_5":  "https://images.unsplash.com/photo-1595090947130-23c5948a2993?w=400",
}

def fetch(url):
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req, timeout=10) as r:
        return Image.open(io.BytesIO(r.read())).convert("RGB")

images = {}
for name, url in SAMPLE_URLS.items():
    try:
        images[name] = fetch(url)
    except Exception as e:
        print(f"FAIL {name}: {e}")
print(f"Geladen: {len(images)} Bilder")

In [ ]:
# Vorschau anzeigen
rows = (len(images)+4)//5
fig, axes = plt.subplots(rows, 5, figsize=(15, 3*rows))
for ax, (name, img) in zip(axes.flat, images.items()):
    ax.imshow(img)
    ax.set_title(name, fontsize=10)
    ax.axis("off")

plt.tight_layout()
plt.show()

### Embedding-Funktion definieren

Wir geben jedem Bild **einen** 384-dimensionalen Vektor (für ViT-S). Wir nehmen den `pooler_output` – das globale Class-Token-Embedding.

In [ ]:
@torch.inference_mode()
def embed(image_or_list):
    """Bettet ein einzelnes Bild oder eine Liste von Bildern ein.
    Gibt einen Tensor (N, hidden_size) zurück."""
    if not isinstance(image_or_list, list):
        image_or_list = [image_or_list]
    inputs = processor(images=image_or_list, return_tensors="pt").to(device)
    outputs = model(**inputs)
    # outputs.pooler_output: (N, hidden_size) – globales Embedding
    # outputs.last_hidden_state: (N, num_patches+specials, hidden_size) – pro Patch
    return outputs.pooler_output.cpu()

# Test mit einem Bild
test_emb = embed(list(images.values())[0])
print(f"Embedding-Form: {test_emb.shape}")
print(f"Norm vor L2: {test_emb.norm():.3f}")

In [ ]:
# Alle Bilder einbetten
names = list(images.keys())
imgs  = list(images.values())

embeddings = embed(imgs)               # (N, 384)
embeddings = F.normalize(embeddings, dim=1)  # L2-normalisieren → Cosine = Dot
print(f"Embeddings-Matrix: {embeddings.shape}")
print(f"Norm nach L2: {embeddings[0].norm():.3f}")  # sollte 1.0 sein

## 4. Embedding-Raum visualisieren

Wir reduzieren die 384 Dimensionen auf 2D mit **UMAP** (oder PCA als schneller Plan B).
Wenn DINOv3 wirklich semantisch generalisiert, sollten Bilder mit ähnlichem Inhalt nahe beieinander landen – ohne dass das Modell je gelabelte Daten gesehen hat.

In [ ]:
# UMAP – falls Installation Probleme macht, fällt der except-Block auf PCA zurück
import umap
from sklearn.decomposition import PCA

reducer_umap = umap.UMAP(n_components=2, n_neighbors=4, min_dist=0.3, random_state=42)
coords_umap = reducer_umap.fit_transform(embeddings.numpy())

coords_pca = PCA(n_components=2).fit_transform(embeddings.numpy())

# Plotten - UMAP
fig, ax = plt.subplots(figsize=(10, 7))
groups = {}
for i, name in enumerate(names):
    grp = name.rsplit("_", 1)[0]   # 'katze_1' → 'katze'
    groups.setdefault(grp, []).append(i)

cmap = plt.get_cmap("tab10")
for j, (grp, idxs) in enumerate(groups.items()):
    color = cmap(j)
    ax.scatter(coords_umap[idxs, 0], coords_umap[idxs, 1], s=200, color=color, label=grp, alpha=0.8)
    for i in idxs:
        ax.annotate(names[i], coords_umap[i] + 0.1, fontsize=9)

ax.set_title(f"DINOv3-Embeddings (2D via UMAP) – semantische Cluster")
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

# Plotten - PCA
fig, ax = plt.subplots(figsize=(10, 7))
groups = {}
for i, name in enumerate(names):
    grp = name.rsplit("_", 1)[0]   # 'katze_1' → 'katze'
    groups.setdefault(grp, []).append(i)

cmap = plt.get_cmap("tab10")
for j, (grp, idxs) in enumerate(groups.items()):
    color = cmap(j)
    ax.scatter(coords_pca[idxs, 0], coords_pca[idxs, 1], s=200, color=color, label=grp, alpha=0.8)
    for i in idxs:
        ax.annotate(names[i], coords_pca[i] + 0.1, fontsize=9)

ax.set_title(f"DINOv3-Embeddings (2D via PCA) – semantische Cluster")
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

Bilder der gleichen Kategorie (z.B. zwei Katzen) sollten in 2D näher beisammen liegen als Bilder unterschiedlicher Kategorien. Das ist erstaunlich, weil DINOv3 nie gelernt hat, was eine Katze ist (Training war self-supervised)

## 5. k-NN-Bildsuche

Klassische Anwendung: Anfrage-Bild → finde die k ähnlichsten anderen Bilder im Datensatz.

In [ ]:
def cosine_similarity_matrix(emb):
    """emb ist L2-normalisiert; Cosine Similarity = Dot-Product."""
    return emb @ emb.T

sim = cosine_similarity_matrix(embeddings)
print(f"Ähnlichkeits-Matrix: {sim.shape}")
print(f"Bereich: [{sim.min():.3f}, {sim.max():.3f}]")

In [ ]:
def show_knn(query_idx, k=4):
    """Zeigt das Anfragebild und die k ähnlichsten anderen."""
    sims = sim[query_idx].clone()
    sims[query_idx] = -1  # sich selbst ausschliessen
    top = sims.topk(k).indices.tolist()

    fig, axes = plt.subplots(1, k+1, figsize=(3*(k+1), 4))
    axes[0].imshow(imgs[query_idx])
    axes[0].set_title(f"Query: {names[query_idx]}", fontsize=11, fontweight="bold")
    axes[0].axis("off")
    for ax, j in zip(axes[1:], top):
        ax.imshow(imgs[j])
        ax.set_title(f"{names[j]}\ncos={sim[query_idx, j]:.3f}", fontsize=10)
        ax.axis("off")
    plt.tight_layout()
    plt.show()

# Mehrere Anfragen demonstrieren
for q in ["katze_1", "auto_1", "pizza_1", "berge_1"]:
    show_knn(names.index(q), k=4)

**Im Unterricht hervorheben:** die Cosine-Similarity-Werte sind konkrete Zahlen, die die Studierenden in ihrer Reflexion notieren sollen. Erwartungswerte:

- Selbe Kategorie: typisch 0.7 – 0.9
- Verschiedene Kategorien: typisch 0.3 – 0.6

Das hilft, ein Gefühl für den Embedding-Raum zu bekommen.

## 6. Vergleich: DINOv3 vs. ImageNet-vortrainiertes ResNet50

Ist DINOv3 wirklich besser als ein klassisch supervised-vortrainiertes Modell?
Wir vergleichen mit ResNet50 von torchvision (auf ImageNet-1K trainiert).

In [ ]:
import torchvision.models as tvm
import torchvision.transforms as T

resnet = tvm.resnet50(weights=tvm.ResNet50_Weights.IMAGENET1K_V2).to(device)
resnet.fc = torch.nn.Identity()  # Klassifikationskopf entfernen → 2048-dim Features
resnet.eval()

resnet_tf = T.Compose([
    T.Resize(256),
    T.CenterCrop(224),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

@torch.inference_mode()
def embed_resnet(image_list):
    batch = torch.stack([resnet_tf(im) for im in image_list]).to(device)
    return resnet(batch).cpu()

resnet_emb = F.normalize(embed_resnet(imgs), dim=1)
print(f"ResNet50-Embeddings: {resnet_emb.shape}")

In [ ]:
# Beide Methoden im selben 2D-Raum vergleichen
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, emb, title in [
    (axes[0], embeddings, "DINOv3 (self-supervised)"),
    (axes[1], resnet_emb, "ResNet50 (ImageNet-supervised)")
]:
    try:
        coords = umap.UMAP(n_components=2, n_neighbors=4, min_dist=0.3,
                           random_state=42).fit_transform(emb.numpy())
    except Exception:
        from sklearn.decomposition import PCA
        coords = PCA(n_components=2).fit_transform(emb.numpy())
    for j, (grp, idxs) in enumerate(groups.items()):
        ax.scatter(coords[idxs, 0], coords[idxs, 1], s=180,
                   color=plt.get_cmap("tab10")(j), label=grp, alpha=0.8)
    ax.set_title(title, fontsize=13)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

**Diskussionspunkt im Unterricht:** Bei diesem kleinen, sauberen Datensatz sehen beide Methoden ähnlich aus. Der Unterschied wird drastisch bei:
- Bildern ausserhalb der ImageNet-Verteilung (medizinische Bilder, Satellitenbilder, Industrie-Defekte)
- Dichten Aufgaben (Segmentierung, Tiefe) – dort ist DINOv3 deutlich besser
- Sehr feinen visuellen Unterschieden

## 7. Bonus: Attention Maps anzeigen

Wir zeigen, was DINOv3 «interessant» findet, indem wir die Cosine-Similarity zwischen einem ausgewählten Patch und allen anderen Patches berechnen. Hohe Ähnlichkeit = das Modell findet diese Region semantisch verwandt.

In [ ]:
@torch.inference_mode()
def patch_features(image):
    """Holt die Patch-Tokens für ein Bild."""
    inputs = processor(images=image, return_tensors="pt").to(device)
    out = model(**inputs)
    # last_hidden_state: (1, num_tokens, hidden_size)
    # Tokens = [CLS] + register tokens (4) + patches
    tokens = out.last_hidden_state[0]
    n_specials = 1 + getattr(model.config, "num_register_tokens", 4)
    patches = tokens[n_specials:]   # (num_patches, hidden)
    return F.normalize(patches, dim=1), inputs["pixel_values"].shape[-1]

def show_patch_similarity(image, query_xy=(0.5, 0.5)):
    """query_xy in (0,1)-Koordinaten – vom welchem Punkt aus berechnen wir die Ähnlichkeit?"""
    patches, img_size = patch_features(image)
    patch_size = 16  # ViT-X/16
    grid = img_size // patch_size

    qx = int(query_xy[0] * grid); qy = int(query_xy[1] * grid)
    q_idx = qy * grid + qx
    sims = (patches @ patches[q_idx]).reshape(grid, grid).cpu().numpy()

    fig, axes = plt.subplots(1, 2, figsize=(11, 5))
    axes[0].imshow(image.resize((img_size, img_size)))
    axes[0].plot(qx*patch_size + patch_size/2, qy*patch_size + patch_size/2,
                  marker="x", markersize=18, markeredgewidth=3, color="red")
    axes[0].set_title(f"Query-Patch (rot) bei {query_xy}")
    axes[0].axis("off")
    im = axes[1].imshow(sims, cmap="viridis")
    axes[1].set_title("Patch-Ähnlichkeit")
    axes[1].axis("off")
    plt.colorbar(im, ax=axes[1], fraction=0.046)
    plt.tight_layout()
    plt.show()

# Klick auf eine Katze – wo findet das Modell ähnliche Patches?
show_patch_similarity(images["katze_1"], query_xy=(0.5, 0.6))
show_patch_similarity(images["auto_1"],  query_xy=(0.5, 0.6))

**Erkenntnis:** Die Patch-Ähnlichkeit zeigt, dass DINOv3 die *Konturen* von Objekten kennt – obwohl es nie gelabelte Segmentierungen gesehen hat. Genau diese Eigenschaft macht es so wertvoll als Backbone für nachgelagerte Aufgaben.

## Take-aways

- **Ein Modell, viele Aufgaben:** Mit dem selben eingefrorenen DINOv3 haben wir Embeddings, k-NN-Suche und semantische Clusterung gemacht – ohne ein einziges Label.
- **Skala zählt:** DINOv3 wurde auf 1.7 Mia. Bildern trainiert. Diese Datenmenge wäre mit Labels nicht bezahlbar.
- **"Eingefrorene" Gewichte ("frozen") funktioniert:** Wir haben das Modell nie nachtrainiert. Für viele Anwendungen reicht das.